# Topology PoC — интерактивный запуск

Каждая ячейка — один логический блок. Запускай последовательно или отдельно для отладки.

**Режимы:**
- `STUB = True` — заглушки вместо LLM, бесплатно, только для проверки пайплайна
- `STUB = False` — реальные вызовы OpenAI API

## 0. Конфигурация

In [ ]:
import sys
from pathlib import Path

# добавляем корень проекта в путь
ROOT = Path(".").resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

MODEL       = "gpt-3.5-turbo"
N_QUESTIONS = 5      # сколько задач GSM8K прогнать
N_AGENTS    = 3      # число агентов (не считая aggregator в star)
STUB        = True   # ← True = без API, False = реальные вызовы

print(f"ROOT: {ROOT}")
print(f"stub={STUB}, model={MODEL}, n_questions={N_QUESTIONS}, n_agents={N_AGENTS}")

## 1. Загрузка датасета GSM8K

In [ ]:
import re

try:
    from datasets import load_dataset
    raw_ds = load_dataset("openai/gsm8k", "main", split=f"test[:{N_QUESTIONS}]")
    questions = [{"question": item["question"], "answer": item["answer"]} for item in raw_ds]
    print("Загружено через HuggingFace datasets")
except Exception as e:
    import urllib.request, json as _json
    print(f"HuggingFace недоступен ({e}), скачиваю с GitHub...")
    url = "https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl"
    with urllib.request.urlopen(url) as r:
        lines = r.read().decode().splitlines()
    questions = [_json.loads(l) for l in lines[:N_QUESTIONS]]

def parse_gt(answer_str: str) -> int:
    return int(re.search(r"####\s*(-?\d+)", answer_str).group(1))

print(f"\nЗагружено задач: {len(questions)}")
print("\n--- Пример ---")
print("Q:", questions[0]["question"][:120], "...")
print("GT:", parse_gt(questions[0]["answer"]))

## 2. Топологии — построение и визуализация

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from topologies.definitions import get_topologies

topologies = get_topologies(n_agents=N_AGENTS)

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
fig.suptitle("Топологии MAS", fontsize=13)

NODE_COLORS = {"task": "#94a3b8", "agent": "#3b82f6"}

for ax, (name, G) in zip(axes, topologies.items()):
    pos = nx.spring_layout(G, seed=42)
    colors = [NODE_COLORS["task"] if n == "task" else NODE_COLORS["agent"] for n in G.nodes]
    nx.draw(G, pos, ax=ax, with_labels=True, node_color=colors,
            node_size=700, font_size=8, font_color="white",
            arrows=True, arrowsize=15, edge_color="#64748b")
    ax.set_title(f"{name}\n{G.number_of_nodes()} nodes, {G.number_of_edges()} edges", fontsize=10)

plt.tight_layout()
plt.savefig(ROOT / "results" / "topologies.png", dpi=120, bbox_inches="tight")
plt.show()

for name, G in topologies.items():
    order = list(nx.topological_sort(G))
    print(f"{name:8s} | nodes: {list(G.nodes)} | topo_order: {order}")

## 3. Метрики графов

In [ ]:
import pandas as pd
from metrics.graph_metrics import TopologyMetrics

engine = TopologyMetrics()
metrics_rows = []

for name, G in topologies.items():
    m = engine.compute(G, name)
    metrics_rows.append({
        "topology": name,
        "diameter": round(m.diameter, 3),
        "avg_degree": round(m.avg_degree, 3),
        "structural_entropy": round(m.structural_entropy, 3),
        "spectral_gap": round(m.spectral_gap, 3),
        "task_centrality": round(m.task_centrality, 3),
    })

metrics_df = pd.DataFrame(metrics_rows).set_index("topology")
metrics_df

## 4. Инициализация агентов

In [ ]:
from mas.agent import Agent, AgentConfig
from mas.runner import MASRunner
from mas.prompts import assign_roles

# roles для N_AGENTS обычных + aggregator (A{N_AGENTS}) в топологии star
roles = assign_roles(N_AGENTS + 1)

def build_agents(graph: nx.DiGraph) -> dict:
    agent_nodes = [n for n in graph.nodes if n != "task"]
    return {
        node: Agent(AgentConfig(
            agent_id=node,
            role=roles.get(node, "solver"),
            model=MODEL,
            stub=STUB,
        ))
        for node in agent_nodes
    }

print("Роли:", roles)
print("STUB =", STUB)

# быстрый smoke-test одного агента
test_agent = Agent(AgentConfig("A0", "solver", stub=True))
response = test_agent.run("What is 2 + 2?", [])
print("\nSmoke-test агента (всегда stub):", response)

## 5. Smoke-test одной топологии на одной задаче

Запусти эту ячейку чтобы убедиться что пайплайн проходит насквозь до сохранения ответа.

In [ ]:
SMOKE_TOPO = "chain"   # ← можно менять: chain / star / full / random
SMOKE_Q    = 0         # ← индекс задачи из датасета

G_smoke  = topologies[SMOKE_TOPO]
agents   = build_agents(G_smoke)
runner   = MASRunner(G_smoke, agents)

q   = questions[SMOKE_Q]
gt  = parse_gt(q["answer"])
out = runner.run(q["question"])

from mas.prompts import parse_answer
pred = parse_answer(out)

print(f"Топология : {SMOKE_TOPO}")
print(f"Вопрос    : {q['question'][:100]}...")
print(f"GT ответ  : {gt}")
print(f"\nОтвет MAS (полный):\n{out}")
print(f"\nПарсинг   : {pred}")
print(f"Верно     : {pred == gt}")

## 6. Полный прогон — все топологии × все задачи

In [ ]:
results = []

for topo_name, G in topologies.items():
    agents = build_agents(G)
    runner = MASRunner(G, agents)
    m = engine.compute(G, topo_name)

    correct, total = 0, len(questions)
    per_question = []

    for i, item in enumerate(questions):
        gt   = parse_gt(item["answer"])
        out  = runner.run(item["question"])
        pred = parse_answer(out)
        ok   = pred is not None and pred == gt
        if ok:
            correct += 1
        per_question.append({"q": i, "gt": gt, "pred": pred, "ok": ok})

    acc = correct / total
    print(f"{topo_name:8s}  accuracy={acc:.2f}  ({correct}/{total})")

    results.append({
        "topology": topo_name,
        "accuracy": acc,
        "per_question": per_question,
        "metrics": {
            "diameter":           m.diameter,
            "avg_degree":         m.avg_degree,
            "structural_entropy": m.structural_entropy,
            "spectral_gap":       m.spectral_gap,
            "task_centrality":    m.task_centrality,
        },
    })

print("\nГотово.")

## 7. Сводная таблица результатов

In [ ]:
summary_rows = []
for r in results:
    summary_rows.append({
        "topology": r["topology"],
        "accuracy": r["accuracy"],
        **{k: round(v, 3) for k, v in r["metrics"].items()},
    })

summary_df = pd.DataFrame(summary_rows).set_index("topology").sort_values("accuracy", ascending=False)
summary_df

## 8. Разбор ошибок — какие задачи провалились

In [ ]:
INSPECT_TOPO = "chain"   # ← какую топологию разбираем

topo_result = next(r for r in results if r["topology"] == INSPECT_TOPO)

errors = [pq for pq in topo_result["per_question"] if not pq["ok"]]
print(f"Ошибки [{INSPECT_TOPO}]: {len(errors)}/{len(topo_result['per_question'])}")
for e in errors:
    q_text = questions[e["q"]]["question"][:80]
    print(f"  Q{e['q']}: GT={e['gt']}, pred={e['pred']} | {q_text}...")

## 9. Сохранение results.json

In [ ]:
import json

out_dir  = ROOT / "results"
out_dir.mkdir(exist_ok=True)
out_path = out_dir / "results.json"

# сохраняем без per_question для совместимости с analysis.ipynb
slim = [{k: v for k, v in r.items() if k != "per_question"} for r in results]
out_path.write_text(json.dumps(slim, indent=2, ensure_ascii=False))

print(f"Сохранено → {out_path}")
print(json.dumps(slim, indent=2, ensure_ascii=False))